# 07 — Model-family screen and submissions

This notebook records lifecycle steps 8 and 9 for a bounded model-family and probability-combination screen. It keeps the existing 29-feature policy, frozen development partition and five predefined folds unchanged. Tree counts are selected inside each outer-training fold; the reserved local test is not reopened.

The screen compares every new standalone not only with the complete incumbent, but also separately with the incumbent Random Forest and histogram-boosting components. A second submission candidate must pass the same accuracy and class-recall gate and disagree with the first on at least 1% of out-of-fold labels.

In [1]:
from pathlib import Path
import json
import sys
import joblib
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
STAGE_DIR = PROJECT_DIR / 'stage-1-pump-it-up'
if not STAGE_DIR.is_dir():
    STAGE_DIR = PROJECT_DIR.parent if PROJECT_DIR.name == 'notebooks' else PROJECT_DIR
    PROJECT_DIR = STAGE_DIR.parent
SRC_DIR = STAGE_DIR / 'src'
RUNTIME_DIR = PROJECT_DIR / '.runtime' / 'gpu-model-screen'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from model_screen_submission import select_submission_candidate_names

screen = joblib.load(RUNTIME_DIR / 'combination-screen.joblib')
summary = screen['summary'].copy()
print(f"Recorded {len(summary)} standalone and probability-combination trials.")

Recorded 115 standalone and probability-combination trials.


In [2]:
display_columns = [
    'mean_accuracy', 'mean_gain', 'fold_wins', 'worst_fold_change',
    'repair_recall', 'non_functional_recall',
    'incumbent_disagreement', 'passes_gate',
]
summary.loc[:, display_columns].head(25).style.format({
    'mean_accuracy': '{:.3%}',
    'mean_gain': '{:+.3%}',
    'worst_fold_change': '{:+.3%}',
    'repair_recall': '{:.3%}',
    'non_functional_recall': '{:.3%}',
    'incumbent_disagreement': '{:.3%}',
})

,mean_accuracy,mean_gain,fold_wins,worst_fold_change,repair_recall,non_functional_recall,incumbent_disagreement,passes_gate
trial,,,,,,,,
60% XGBoost depth 8 [current one-hot] + 40% Random Forest,81.555%,+0.154%,4,-0.042%,34.309%,78.312%,2.157%,True
60% equal XGBoost depth 8 seed bag + 40% Random Forest,81.551%,+0.149%,4,+0.000%,34.251%,78.268%,2.115%,True
60% XGBoost variant bag + 40% Random Forest,81.534%,+0.133%,4,-0.063%,34.280%,78.367%,2.111%,True
60% XGBoost depth 8 [current one-hot] seed 20260823 + 40% Random Forest,81.519%,+0.118%,5,+0.021%,34.338%,78.192%,2.210%,False
50% XGBoost depth 8 [current one-hot] + 50% Random Forest,81.519%,+0.118%,3,-0.042%,34.714%,78.498%,1.949%,True
60% XGBoost depth 11 conservative [current one-hot] + 40% Random Forest,81.511%,+0.109%,4,-0.032%,34.453%,78.307%,2.285%,True
60% XGBoost depth 8 [current one-hot] seed 20260822 + 40% Random Forest,81.500%,+0.099%,4,-0.053%,34.077%,78.285%,2.212%,False
50% XGBoost depth 8 [current one-hot] seed 20260822 + 50% Random Forest,81.488%,+0.086%,4,-0.095%,34.772%,78.422%,1.978%,False
50% XGBoost depth 11 conservative [current one-hot] + 50% Random Forest,81.471%,+0.069%,4,-0.095%,35.032%,78.389%,2.037%,False


In [3]:
standalone_names = [
    name for name, recipe in screen['recipes'].items()
    if len(recipe) == 1 and recipe[0][0] == name
]
print('Standalone models and incumbent components:')
summary.loc[summary.index.intersection(standalone_names), display_columns].sort_values(
    'mean_accuracy', ascending=False
).style.format({
    'mean_accuracy': '{:.3%}', 'mean_gain': '{:+.3%}',
    'worst_fold_change': '{:+.3%}', 'repair_recall': '{:.3%}',
    'non_functional_recall': '{:.3%}', 'incumbent_disagreement': '{:.3%}',
})

Standalone models and incumbent components:


,mean_accuracy,mean_gain,fold_wins,worst_fold_change,repair_recall,non_functional_recall,incumbent_disagreement,passes_gate
trial,,,,,,,,
incumbent 50% Random Forest + 50% boosting,81.402%,+0.000%,0,+0.000%,33.643%,78.153%,0.000%,False
XGBoost lossguide 64 [current one-hot],80.947%,-0.455%,0,-0.800%,32.774%,77.156%,5.128%,False
XGBoost depth 11 conservative [current one-hot],80.934%,-0.467%,0,-0.768%,31.442%,77.124%,5.107%,False
XGBoost depth 8 [current one-hot] seed 20260822,80.924%,-0.478%,0,-0.673%,31.124%,76.998%,5.455%,False
XGBoost depth 8 [current one-hot],80.909%,-0.492%,0,-0.800%,31.442%,77.178%,5.452%,False
XGBoost depth 8 [current one-hot] seed 20260823,80.844%,-0.558%,0,-0.852%,31.182%,76.866%,5.442%,False
LightGBM leaves 127 conservative [current one-hot],80.678%,-0.724%,0,-1.178%,31.095%,76.833%,5.669%,False
Random Forest,80.591%,-0.810%,0,-1.010%,36.856%,78.898%,5.236%,False
LightGBM leaves 63 [current one-hot],80.589%,-0.812%,0,-1.231%,30.719%,76.587%,5.888%,False


In [4]:
component_pair_names = [
    name for name, recipe in screen['recipes'].items()
    if len(recipe) == 2
    and any(component in {'Random Forest', 'histogram boosting'} for component, _ in recipe)
    and name != screen['incumbent_name']
]
print('Strongest direct pairings with an incumbent component:')
summary.loc[summary.index.intersection(component_pair_names), display_columns].sort_values(
    'mean_accuracy', ascending=False
).head(20).style.format({
    'mean_accuracy': '{:.3%}', 'mean_gain': '{:+.3%}',
    'worst_fold_change': '{:+.3%}', 'repair_recall': '{:.3%}',
    'non_functional_recall': '{:.3%}', 'incumbent_disagreement': '{:.3%}',
})

Strongest direct pairings with an incumbent component:


,mean_accuracy,mean_gain,fold_wins,worst_fold_change,repair_recall,non_functional_recall,incumbent_disagreement,passes_gate
trial,,,,,,,,
60% XGBoost depth 8 [current one-hot] + 40% Random Forest,81.555%,+0.154%,4,-0.042%,34.309%,78.312%,2.157%,True
60% equal XGBoost depth 8 seed bag + 40% Random Forest,81.551%,+0.149%,4,+0.000%,34.251%,78.268%,2.115%,True
60% XGBoost variant bag + 40% Random Forest,81.534%,+0.133%,4,-0.063%,34.280%,78.367%,2.111%,True
60% XGBoost depth 8 [current one-hot] seed 20260823 + 40% Random Forest,81.519%,+0.118%,5,+0.021%,34.338%,78.192%,2.210%,False
50% XGBoost depth 8 [current one-hot] + 50% Random Forest,81.519%,+0.118%,3,-0.042%,34.714%,78.498%,1.949%,True
60% XGBoost depth 11 conservative [current one-hot] + 40% Random Forest,81.511%,+0.109%,4,-0.032%,34.453%,78.307%,2.285%,True
60% XGBoost depth 8 [current one-hot] seed 20260822 + 40% Random Forest,81.500%,+0.099%,4,-0.053%,34.077%,78.285%,2.212%,False
50% XGBoost depth 8 [current one-hot] seed 20260822 + 50% Random Forest,81.488%,+0.086%,4,-0.095%,34.772%,78.422%,1.978%,False
50% XGBoost depth 11 conservative [current one-hot] + 50% Random Forest,81.471%,+0.069%,4,-0.095%,35.032%,78.389%,2.037%,False


In [5]:
selected_names = select_submission_candidate_names(screen)
passing = summary.loc[summary['passes_gate'], display_columns]
print(f"Gate-passers: {len(passing)}")
print('Selected submission candidates:', selected_names or 'none')
if len(selected_names) == 2:
    first = screen['evaluations'][selected_names[0]].out_of_fold_probabilities.to_numpy().argmax(axis=1)
    second = screen['evaluations'][selected_names[1]].out_of_fold_probabilities.to_numpy().argmax(axis=1)
    print(f"Selected-candidate OOF label disagreement: {np.mean(first != second):.3%}")
passing.style.format({
    'mean_accuracy': '{:.3%}', 'mean_gain': '{:+.3%}',
    'worst_fold_change': '{:+.3%}', 'repair_recall': '{:.3%}',
    'non_functional_recall': '{:.3%}', 'incumbent_disagreement': '{:.3%}',
})

Gate-passers: 5
Selected submission candidates: ['60% XGBoost depth 8 [current one-hot] + 40% Random Forest', '60% XGBoost depth 11 conservative [current one-hot] + 40% Random Forest']
Selected-candidate OOF label disagreement: 1.307%


,mean_accuracy,mean_gain,fold_wins,worst_fold_change,repair_recall,non_functional_recall,incumbent_disagreement,passes_gate
trial,,,,,,,,
60% XGBoost depth 8 [current one-hot] + 40% Random Forest,81.555%,+0.154%,4,-0.042%,34.309%,78.312%,2.157%,True
60% equal XGBoost depth 8 seed bag + 40% Random Forest,81.551%,+0.149%,4,+0.000%,34.251%,78.268%,2.115%,True
60% XGBoost variant bag + 40% Random Forest,81.534%,+0.133%,4,-0.063%,34.280%,78.367%,2.111%,True
50% XGBoost depth 8 [current one-hot] + 50% Random Forest,81.519%,+0.118%,3,-0.042%,34.714%,78.498%,1.949%,True
60% XGBoost depth 11 conservative [current one-hot] + 40% Random Forest,81.511%,+0.109%,4,-0.032%,34.453%,78.307%,2.285%,True


In [6]:
report_path = RUNTIME_DIR / 'selected-submissions.json'
submission_records = json.loads(report_path.read_text(encoding='utf-8')) if report_path.exists() else []
if submission_records:
    for record in submission_records:
        print(f"#{record['rank']} {record['candidate']}")
        print(f"  rows={record['rows']:,}; sha256={record['sha256']}")
        print(f"  class shares={record['competition_class_shares']}")
else:
    print('No competition CSV was generated.')

#1 60% XGBoost depth 8 [current one-hot] + 40% Random Forest
  rows=14,850; sha256=34740dd7972f4a44cdfc747bb314de68cc24fc92d0cf8ff708196a67d7d8a8b6
  class shares={'functional': 0.6053872053872054, 'functional needs repair': 0.03750841750841751, 'non functional': 0.3571043771043771}
#2 60% XGBoost depth 11 conservative [current one-hot] + 40% Random Forest
  rows=14,850; sha256=69bf072d7662ab1a3a5024ecb38ae4eb286d4e6b434c6b26eafc3c024ba89071
  class shares={'functional': 0.6041750841750841, 'functional needs repair': 0.03838383838383838, 'non functional': 0.35744107744107745}
